# Exercise 12 - Residual Networks

The goal of this exercise is to create and run your own residual network starting from a given network with a simple but deep architecture.

We start with a simple but quite deep network of 18 x 3x3 convolutional layers, each with batch normalization in front of a ReLU activation function. Batch normalization should keep the activations and weights under control in this deep network. Run the following code to train this network on CIFAR-10.
- Run this exercise using the **T4 GPU**

### Step 1:
Download and save CIFAR-10 dataset from Kaggle

In [ ]:
import os
if not os.path.exists('./cifar10-python.zip'):
  !curl -L -o cifar10-python.zip https://www.kaggle.com/api/v1/datasets/download/pankrzysiu/cifar10-python

### Step 2:
Perform the following pre-processing operations for dataset preparation
- Unzip the dataset
- Extract batches of training and test images
- Prepreprocess images to conform to Keras channels-last format (num_images, 32, 32, 3)
- Create x_train, y_train, x_test and y_test arrays
- Print out the shapes of these arrays

In [ ]:
import os
import zipfile
import pickle
import numpy as np
from tensorflow.keras.utils import to_categorical

def extract_cifar10_zip(zip_path, extract_to):
    """Extracts the local zip file into the target directory."""
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Extraction complete.")

    # Locate the folder inside the extracted directory
    # Depending on how it was zipped, files might be in the root or a subfolder
    for root, dirs, files in os.walk(extract_to):
        if "data_batch_1" in files:
            return root
    raise FileNotFoundError("Could not find CIFAR-10 data batches in the extracted files.")

def load_pickle_batch(file_path):
    """Loads a single un-pickled batch file from disk."""
    with open(file_path, 'rb') as f:
        # encoding='latin1' ensures Python 3 compatibility with byte streams
        batch_dict = pickle.load(f, encoding='latin1')
    return batch_dict

def preprocess_images(raw_data):
    """Converts raw flat features into channels-last normalized images."""
    # CIFAR stores channels-first: 3 channels by 1024 pixels (32x32)
    images = raw_data.reshape(-1, 3, 32, 32)
    # Transpose to Keras channels-last layout: (num_images, 32, 32, 3)
    images = images.transpose(0, 2, 3, 1)
    # Cast to float and scale pixel values down to [0.0, 1.0]
    return images.astype('float32') / 255.0

def process_dataset(data_dir):
    """Loops through extracted files to build training and testing sets."""
    x_train_list = []
    y_train_list = []

    # 1. Compile the 5 training data batches
    for i in range(1, 6):
        batch_file = os.path.join(data_dir, f"data_batch_{i}")
        batch = load_pickle_batch(batch_file)
        x_train_list.append(batch['data'])
        y_train_list.extend(batch['labels'])

    x_train_raw = np.concatenate(x_train_list, axis=0)
    y_train = np.array(y_train_list)

    # 2. Compile the single testing batch
    test_file = os.path.join(data_dir, "test_batch")
    test_batch = load_pickle_batch(test_file)
    x_test_raw = test_batch['data']
    y_test = np.array(test_batch['labels'])

    # 3. Shape and scale the image arrays
    x_train = preprocess_images(x_train_raw)
    x_test = preprocess_images(x_test_raw)

    # 4. Transform scalar labels to 10-class one-hot vectors
    y_train = to_categorical(y_train, num_classes=10)
    y_test = to_categorical(y_test, num_classes=10)

    return (x_train, y_train), (x_test, y_test)

if __name__ == "__main__":
    # Ensure this points directly to your local zip archive
    ZIP_FILE_PATH = "./cifar10-python.zip"
    OUTPUT_DIRECTORY = "./extracted_cifar10"

    if not os.path.exists(ZIP_FILE_PATH):
        print(f"Error: Please place '{ZIP_FILE_PATH}' in this directory before running.")
    else:
        # Run conversion pipeline
        data_batch_folder = extract_cifar10_zip(ZIP_FILE_PATH, OUTPUT_DIRECTORY)
        (x_train, y_train), (x_test, y_test) = process_dataset(data_batch_folder)

        # Verify output formats
        print("\n--- Processing Results ---")
        print(f"x_train array: {x_train.shape} | Type: {x_train.dtype}")
        print(f"y_train array: {y_train.shape} | Type: {y_train.dtype}")
        print(f"x_test array:  {x_test.shape} | Type: {x_test.dtype}")
        print(f"y_test array:  {y_test.shape} | Type: {y_test.dtype}")

In [ ]:
import tensorflow
from tensorflow.keras.layers import Input, Dense, Activation, Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling2D, add
from tensorflow.keras.models import Model
from tensorflow.keras.utils  import plot_model


tensorflow.keras.backend.clear_session()

def conv_bn_relu(IN, n_features, size, padding='same', strides=(1, 1)):
    L1 = Conv2D(n_features, size, strides=strides, padding=padding, use_bias=False)(IN)
    L2 = BatchNormalization(scale=False)(L1)
    L3 = Activation('relu')(L2)
    return L3

input_img = Input(shape=(32, 32, 3))
IN = input_img

C1 = conv_bn_relu(IN, 16, (3, 3))
C2 = conv_bn_relu(C1, 16, (3, 3))
C3 = conv_bn_relu(C2, 16, (3, 3))
C4 = conv_bn_relu(C3, 16, (3, 3))
C5 = conv_bn_relu(C4, 16, (3, 3))
C6 = conv_bn_relu(C5, 16, (3, 3))

C7 = conv_bn_relu(C6, 32, (3, 3), strides=(2, 2))
C8 = conv_bn_relu(C7, 32, (3, 3))
C9 = conv_bn_relu(C8, 32, (3, 3))
C10 = conv_bn_relu(C9, 32, (3, 3))
C11 = conv_bn_relu(C10, 32, (3, 3))
C12 = conv_bn_relu(C11, 32, (3, 3))

C13 = conv_bn_relu(C12, 64, (3, 3), strides=(2, 2))
C14 = conv_bn_relu(C13, 64, (3, 3))
C15 = conv_bn_relu(C14, 64, (3, 3))
C16 = conv_bn_relu(C15, 64, (3, 3))
C17 = conv_bn_relu(C16, 64, (3, 3))
C18 = conv_bn_relu(C17, 64, (3, 3))

C = Conv2D(10, (1, 1), strides=(1, 1))(C18)
P = GlobalAveragePooling2D(name='avg_pool')(C)
S = Activation('softmax', name='softmax')(P)

model = Model(inputs=input_img, outputs=S)
model.summary()

plot_model(model, "deep_convolution.png", dpi=100, show_shapes=True)

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(x_train, y_train,
          batch_size=32, epochs=10,
          validation_data=(x_test, y_test), shuffle=True);

Now create a residual network based on the model above by adding a shortcut around each pair of convolution filters, as shown on the slides, and compare the results (the accuracy and whether the network is over-fitting the data) to the non-residual version.

In [ ]:
#

#### Here is our answer. Do not run the cell below unless you want to see the answer we provide!

In [ ]:
from tensorflow.keras.layers import add
from tensorflow.keras.utils  import plot_model

def conv_bn_relu(IN, n_features, size, padding='same', strides=(1, 1)):
    L1 = Conv2D(n_features, size, strides=strides, padding=padding, use_bias=False)(IN)
    L2 = BatchNormalization(scale=False)(L1)
    L3 = Activation('relu')(L2)
    return L3

def residual_block(IN, n_features, strides=(1, 1)):
    if strides == (1, 1):
        if IN.shape[2] == n_features:
            shortcut = IN
        else:
            shortcut = Conv2D(n_features, (1, 1))(IN)
    else:
        shortcut = MaxPooling2D(pool_size=strides[0], padding='same')(IN)
        shortcut = Conv2D(n_features, (1, 1), strides=(1, 1))(shortcut)
    C1 = conv_bn_relu(IN, n_features, (3, 3), strides=strides)
    C2 = conv_bn_relu(C1, n_features, (3, 3))
    return add([C2, shortcut])

input_img = Input(shape=(32, 32, 3))
IN = input_img

B1 = residual_block(IN, 16)
B2 = residual_block(B1, 16)
B3 = residual_block(B2, 16)

B4 = residual_block(B3, 32, strides=(2, 2))
B5 = residual_block(B4, 32)
B6 = residual_block(B5, 32)

B7 = residual_block(B6, 64, strides=(2, 2))
B8 = residual_block(B7, 64)
B9 = residual_block(B8, 64)

C = Conv2D(10, (1, 1), strides=(1, 1))(B9)
P = GlobalAveragePooling2D(name='avg_pool')(C)
S = Activation('softmax', name='softmax')(P)

model = Model(inputs=input_img, outputs=S)
model.summary()
plot_model(model, "mini_resnet.png", dpi=100, show_shapes=True)

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(x_train, y_train,
          batch_size=32, epochs=10,
          validation_data=(x_test, y_test), shuffle=True);